Train Monthly Model — Live Production Data
Trains Prophet on monthly gold data, evaluates against baseline, saves 3-month forecast.

**Input**: gold/erp/battery/phase1_overall_monthly_live.parquet
**Output**: printed 3-month forecast with confidence intervals

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

In [0]:
gold_monthly = read_gold(blob_service, "live/battery/data/phase1_overall_monthly_live.parquet")
gold_monthly["month_start"] = pd.to_datetime(gold_monthly["month_start"])

monthly_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(
    columns={"month_start": "ds", "total_units_sold": "y"}
)

print(monthly_prophet_df.tail(5))

In [0]:
train_monthly = monthly_prophet_df.iloc[:-3]
test_monthly = monthly_prophet_df.iloc[-3:]

m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_test.fit(train_monthly)

future_test = m_test.make_future_dataframe(periods=3, freq="MS")
forecast_test = m_test.predict(future_test)
test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

print(f"Monthly Prophet WAPE: {wape(test_monthly['y'].values, test_preds):.3%}")

naive_pred = train_monthly["y"].tail(3).mean()
print(f"Monthly Naive Baseline WAPE: {wape(test_monthly['y'].values, [naive_pred]*3):.3%}")

In [0]:
m_final_capped = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
m_final_capped.fit(monthly_prophet_df)

future_final_capped = m_final_capped.make_future_dataframe(periods=3, freq="MS")
forecast_final_capped = m_final_capped.predict(future_final_capped)
forecast_final_capped[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final_capped[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

print(forecast_final_capped[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [0]:
gold_monthly["month_num"] = gold_monthly["month_start"].dt.month
gold_monthly["year"] = gold_monthly["month_start"].dt.year

aso_history = gold_monthly[gold_monthly["month_num"].isin([8, 9, 10])].sort_values("month_start")
print(aso_history[["month_start", "total_units_sold"]])